# Deep Regret Analytic Generative Adversarial Network (DRAGAN) <BR> Applied to CelebA Dataset and 64 x 64 images

Antonio Esteves @ UMinho, Jun 2024

In [ ]:
import os
import shutil
import numpy                  as     np
import PIL.Image              as     Image
from   pathlib                import Path
from   natsort                import natsorted
import matplotlib.pyplot      as     plt
import wandb
import time
import yaml

import torch
import torchvision
from   torch.autograd         import grad
from   torch.autograd         import Variable
import torch.nn               as     nn
import torchvision.transforms as     transforms
from   torch.utils.data       import DataLoader, Dataset
from   torchvision.utils      import save_image, make_grid

## Configuration

In [ ]:
LOAD_TRAINED_MODEL        = False
SKIP_TRAIN_MODEL          = False

CONFIG_FILE = 'config/config_dragan_celeba_64x64_v4_02.yaml'

with open(CONFIG_FILE, 'r') as file:
    try:
        config = yaml.safe_load(file)
    except yaml.YAMLError as exc:
        print(exc)

In [ ]:
if (config['crop_size'] == 'None'):
    config['crop_size'] = None

In [ ]:
print('parameters:')
for key, value in config.items():
    print(f'\t{key}: {value}')

## Initializations and create necessary folders

In [ ]:
# Setup device agnostic code

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f'Using {device} for computing')

train_dir = Path(config["dataset_path"])

# Location where we will save here the images generated during DRAGAN training
RESULTS_PATH=f'results/{config["experiment_name"]}'
os.makedirs(RESULTS_PATH, exist_ok=True)

# Location where the trained models will be saved
MODELS_PATH      = os.path.join(os.getcwd(), 'models')
os.makedirs(MODELS_PATH, exist_ok=True)

## Login into Weights & Bias

In [ ]:
wandb.login()

## Track metadata and hyperparameters with Weights & Bias

Define the experiment: the hyperparameters, the dataset and model name. This information will be stored in a `config` dictionary.

In [ ]:
config_wandb = config

wandb.init(project='GAN_CLOTHES', entity='ajesteves', config=config_wandb)

## Utility functions

In [ ]:
def time_format(seconds: int) -> str:
    if seconds is not None:
        seconds = int(seconds)
        d = seconds // (3600 * 24)
        h = seconds // 3600 % 24
        m = seconds % 3600 // 60
        s = seconds % 3600 % 60
        if d > 0:
            return '{:02d}D {:02d}H {:02d}m {:02d}s'.format(d, h, m, s)
        elif h > 0:
            return '{:02d}H {:02d}m {:02d}s'.format(h, m, s)
        elif m > 0:
            return '{:02d}m {:02d}s'.format(m, s)
        elif s > 0:
            return '{:02d}s'.format(s)
    return '-'


def save_checkpoint(state, save_path, is_best=False, max_keep=None):
    # save checkpoint
    torch.save(state, save_path)

    # deal with max_keep
    save_dir = os.path.dirname(save_path)
    list_path = os.path.join(save_dir, 'latest_checkpoint')

    save_path = os.path.basename(save_path)
    if os.path.exists(list_path):
        with open(list_path) as f:
            ckpt_list = f.readlines()
            ckpt_list = [save_path + '\n'] + ckpt_list
    else:
        ckpt_list = [save_path + '\n']

    if max_keep is not None:
        for ckpt in ckpt_list[max_keep:]:
            ckpt = os.path.join(save_dir, ckpt[:-1])
            if os.path.exists(ckpt):
                os.remove(ckpt)
        ckpt_list[max_keep:] = []

    with open(list_path, 'w') as f:
        f.writelines(ckpt_list)

    # copy best
    if is_best:
        shutil.copyfile(save_path, os.path.join(save_dir, 'best_model.ckpt'))


def load_checkpoint(ckpt_dir_or_file, map_location=None, load_best=False):
    if os.path.isdir(ckpt_dir_or_file):
        if load_best:
            ckpt_path = os.path.join(ckpt_dir_or_file, 'best_model.ckpt')
        else:
            with open(os.path.join(ckpt_dir_or_file, 'latest_checkpoint')) as f:
                ckpt_path = os.path.join(ckpt_dir_or_file, f.readline()[:-1])
    else:
        ckpt_path = ckpt_dir_or_file
    ckpt = torch.load(ckpt_path, map_location=map_location)
    print(' [*] Loading checkpoint from %s succeed!' % ckpt_path)
    return ckpt

## Calculate the gradient penalty

In [ ]:
def gradient_penalty(x, discriminator, k):

    # interpolation
    shape = [x.size(0)] + [1] * (x.dim() - 1)
    alpha = torch.rand(shape).to(device)
    beta  = torch.rand(x.size()).to(device)

    y     = x + 0.5 * x.std() * beta
    z     = x + alpha * (y - x)

    # gradient penalty
    z               = z.to(device)
    z.requires_grad = True

    o  = discriminator(z)
    g  = grad(o, z, grad_outputs=torch.ones(o.size()).to(device), create_graph=True)[0].view(z.size(0), -1)
    gp = ((g.norm(p=2, dim=1) - k)**2).mean()

    return gp

## Layer Normalization

In [ ]:
class LayerNorm(nn.Module):

    def __init__(self, num_features, eps=1e-5, affine=True):
        super(LayerNorm, self).__init__()
        self.num_features = num_features
        self.affine       = affine
        self.eps          = eps

        if self.affine:
            self.gamma = nn.Parameter(torch.Tensor(num_features).uniform_())
            self.beta  = nn.Parameter(torch.zeros(num_features))

    def forward(self, x):
        # This implementation is too slow!!!

        shape = [-1] + [1] * (x.dim() - 1)
        mean  = x.view(x.size(0), -1).mean(1).view(*shape)
        std   = x.view(x.size(0), -1).std(1).view(*shape)
        y     = (x - mean) / (std + self.eps)
        if self.affine:
            shape = [1, -1] + [1] * (x.dim() - 2)
            y     = self.gamma.view(*shape) * y + self.beta.view(*shape)
        return y


## Generator Model

In [ ]:
class Generator(nn.Module):

    def __init__(self, in_dim, dim=64):
        super(Generator, self).__init__()

        def dconv_bn_relu(in_dim, out_dim):
            return nn.Sequential(
                nn.ConvTranspose2d(in_dim, out_dim, 5, 2,
                                   padding=2, output_padding=1, bias=False),
                nn.BatchNorm2d(out_dim),
                nn.ReLU())

        self.l1 = nn.Sequential(
            nn.Linear(in_dim, dim * 8 * 4 * 4, bias=False),
            nn.BatchNorm1d(dim * 8 * 4 * 4),
            nn.ReLU())
        self.l2_5 = nn.Sequential(
            dconv_bn_relu(dim * 8, dim * 4),
            dconv_bn_relu(dim * 4, dim * 2),
            dconv_bn_relu(dim * 2, dim),
            nn.ConvTranspose2d(dim, 3, 5, 2, padding=2, output_padding=1),
            nn.Tanh())

    def forward(self, x):
        y = self.l1(x)
        y = y.view(y.size(0), -1, 4, 4)
        y = self.l2_5(y)
        return y


## Discriminator Model

In [ ]:

class Discriminator(nn.Module):

    def __init__(self, in_dim, dim=64):
        super(Discriminator, self).__init__()

        def conv_ln_lrelu(in_dim, out_dim):
            return nn.Sequential(
                nn.Conv2d(in_dim, out_dim, 5, 2, 2),
                # Since there is no effective implementation of LayerNorm,
                # we use InstanceNorm2d instead of LayerNorm here.
                nn.InstanceNorm2d(out_dim, affine=True),
                nn.LeakyReLU(0.2))

        self.ls = nn.Sequential(
            nn.Conv2d(in_dim, dim, 5, 2, 2), nn.LeakyReLU(0.2),
            conv_ln_lrelu(dim, dim * 2),
            conv_ln_lrelu(dim * 2, dim * 4),
            conv_ln_lrelu(dim * 4, dim * 8),
            nn.Conv2d(dim * 8, 1, 4))

    def forward(self, x):
        y = self.ls(x)
        y = y.view(-1)
        return y

## Create a custom Dataset from the images in a folder

In [ ]:
class CustomDataSet(Dataset):

    def __init__(self, root_dir, transform):
        self.root_dir     = root_dir
        self.transform    = transform
        self.all_images   = os.listdir(root_dir)
        self.total_images = natsorted(self.all_images)

    def __len__(self):
        return len(self.total_images)

    def __getitem__(self, idx):
        img_loc      = os.path.join(self.root_dir, self.total_images[idx])
        image        = Image.open(img_loc).convert("RGB")
        tensor_image = self.transform(image)
        return tensor_image

### Define the transformation that will be applied to the images

- convert the images to tensors
- crop the images
- resize the images
- normalize the images. 

### Instantiate a Custom Dataset

### Create a training DataLoader

In [ ]:
def get_loader(crop_size, image_size, batch_size, dataset_train_dir):

    if crop_size != None:
        offset_height = (218 - crop_size) // 2
        offset_width  = (178 - crop_size) // 2
        crop = lambda x: x[:, offset_height:offset_height + crop_size, offset_width:offset_width + crop_size]

        train_transform = transforms.Compose(
            [
                transforms.ToTensor(),
                transforms.Lambda(crop),
                transforms.Resize(size=(config["image_size"], config["image_size"]), antialias=True),
                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
            ]
        )
    else:
        train_transform = transforms.Compose(
            [
                transforms.ToTensor(),
                transforms.Resize(size=(config["image_size"], config["image_size"]), antialias=True),
                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
            ]
        )


    train_data = CustomDataSet(
        root_dir  = train_dir,
        transform = train_transform,
    )

    train_loader = DataLoader(
        dataset     = train_data,
        batch_size  = config["batch_size"],
        shuffle     = True,
        drop_last   = True,
        num_workers = 4,
        pin_memory  = True,
    )
    return train_loader, train_data

## Let us check if everything works fine and display a few real images.

In [ ]:
def check_dataloader(crop_size, image_size, batch_size, dataset_train_dir):
    NR, NC    = 3, 3
    loader, _ = get_loader(crop_size, image_size, batch_size, dataset_train_dir)
    imgs      = next(iter(loader))
    print(f'Batch of images shape: {imgs.shape}')   # BS, Ch, H, W

    if NR*NC > imgs.shape[0]:
        NR = 2
        if NR*NC > imgs.shape[0]:
            NR = 1
            if NR*NC > imgs.shape[0]:
                NC = 2

    _, ax    = plt.subplots(NR, NC, figsize=(3*NC,3*NR))
    plt.suptitle(
        'Some real images of {config["dataset"]} dataset',
        fontsize=15,
        fontweight='bold'
    )

    index = 0
    for r in range(NR):
        for c in range(NC):
            index += 1
            if NR==1:
                ax[c].imshow((imgs[index].permute(1,2,0)+1)/2) 
            else:
                ax[r][c].imshow((imgs[index].permute(1,2,0)+1)/2) 

In [ ]:
check_dataloader(
    crop_size         = config["crop_size"],
    image_size        = config["image_size"], 
    batch_size        = config["batch_size"], 
    dataset_train_dir = train_dir,
)

## Functions to save and load the models to/from file

In [ ]:
def save_model_and_results(
        discriminator,
        generator,
        d_optimizer,
        g_optimizer,
        results,
        epoch,
        hyperparameters,
        file_name
    ):
    results_to_save = {
        'discriminator':   discriminator.state_dict(),
        'generator':       generator.state_dict(),
        'd_optimizer':     d_optimizer.state_dict(),
        'g_optimizer':     g_optimizer.state_dict(),
        'results':         results,
        'epoch':           epoch,
        'hyperparameters': hyperparameters,
    }

    torch.save(
        results_to_save,
        file_name,
    )

In [ ]:
def load_model(discriminator, generator, d_optimizer, g_optimizer, file_name, device):
    '''
    Given instances of the generator and discriminator models, loads from file 'file_name':
    (i)   the weights of both models,
    (ii)  the optimizers state,
    (iii) the results obtained during model training and
    (iv)  the training hyperparameters used to train the models,
    and put the models on 'device'.

    Returns the loaded results and the loaded hyperparameters.
    '''

    results_loaded = torch.load(file_name)

    discriminator.load_state_dict(results_loaded['discriminator'])
    discriminator.to(device)

    generator.load_state_dict(results_loaded['generator'])
    generator.to(device)

    d_optimizer.load_state_dict(results_loaded['d_optimizer'])
    g_optimizer.load_state_dict(results_loaded['g_optimizer'])

    # Returns the saved results and the saved hyperparameters
    return results_loaded['results'], results_loaded['epoch'], results_loaded['hyperparameters']

## Instantiate the Models and select the loss function and optimizers

In [ ]:
discriminator = Discriminator(3, dim=config["d_hidden_dim"]).to(device)
generator     = Generator(config["z_dim"], dim=config["g_hidden_dim"]).to(device)

bce           = nn.BCEWithLogitsLoss().to(device)

d_optimizer = torch.optim.Adam(
    discriminator.parameters(),
    lr    = config["lr"],
    betas = (config["beta1"], config["beta2"])
)
g_optimizer = torch.optim.Adam(
    generator.parameters(),
    lr    = config["lr"],
    betas = (config["beta1"], config["beta2"])
)

## Print the generator model summary

In [ ]:
from torchinfo import summary

aux_data = torch.randn(config['batch_size'], config["z_dim"], device=device)
summary(
    generator,
    input_data   = aux_data,
    col_width    = 16,
    col_names    = ["kernel_size", "output_size", "num_params"],
    row_settings = ["var_names"],
)

## Print the discriminator model summary

In [ ]:
from torchinfo import summary

aux_data = torch.randn(
    (
    config['batch_size'],
    config['channels'], 
    config['image_size'],
    config['image_size']
    )).to(device)

summary(
    discriminator,
    input_data   = aux_data,
    col_width    = 16,
    col_names    = ["kernel_size", "output_size", "num_params"],
    row_settings = ["var_names"],
)

## Train the DRAGAN Model

In [ ]:
def train_dragan(
        discriminator,
        generator,
        optimizer_discriminator,
        optimizer_G,
        dataloader,
        start_epoch,
        config,
        results,
        device,
    ):

    # training loop ............................................................

    ref_batch_size = 25

    # Fixed input batch of random vectors to used to evaluate the generation
    # quality during training
    z_sample = torch.randn(ref_batch_size, config["z_dim"], device=device)

    for epoch in range(start_epoch, config["epochs"]):

        ts = time.time()
        for i, imgs in enumerate(train_loader):

            # Put the model in training mode
            generator.train()

            imgs     = Variable(imgs)
            bs       = imgs.size(0)

            # Random input noise vector
            z        = Variable(torch.randn(bs, config["z_dim"]))
            # Real labels('1..1')
            r_labels = Variable(torch.ones(bs))
            # Fake labels ('0...0')
            f_labels = Variable(torch.zeros(bs))

            imgs, z, r_labels, f_labels = [imgs.to(device), z.to(device), r_labels.to(device), f_labels.to(device)]

            # Gnerate fake images from noise vector
            f_imgs = generator(z)

            # ...................................................
            # Train the discriminator
            # ...................................................

            r_logits = discriminator(imgs)
            f_logits = discriminator(f_imgs.detach())
            d_r_loss = bce(r_logits, r_labels)
            d_f_loss = bce(f_logits, f_labels)
            gp       = gradient_penalty(imgs.data, discriminator, config['k'])
            d_loss   = d_r_loss + d_f_loss + gp * config["lambda_gp"]

            discriminator.zero_grad()
            d_loss.backward()
            d_optimizer.step()

            # ...................................................
            # Train the generator
            # ...................................................

            f_logits = discriminator(f_imgs)
            g_loss   = bce(f_logits, r_labels)

            discriminator.zero_grad()
            generator.zero_grad()
            g_loss.backward()
            g_optimizer.step()

            # Save the results in a dictionary ......................................
            results["d_loss"].append(d_loss.item())
            results["d_r_loss"].append(d_r_loss.item())
            results["d_f_loss"].append(d_f_loss.item())
            results["g_loss"].append(g_loss.item())
            results["gp"].append(gp.data.item())\

            # Print progress metrics and save them to W&B ..........................
            if i % config["log_interval"] == 0:

                mean_d_loss      = np.mean(results["d_loss"][-config["log_interval"]:])
                mean_d_r_loss    = np.mean(results["d_r_loss"][-config["log_interval"]:])
                mean_d_f_loss    = np.mean(results["d_f_loss"][-config["log_interval"]:])
                mean_g_loss      = np.mean(results["g_loss"][-config["log_interval"]:])
                mean_gp          = np.mean(results["gp"][-config["log_interval"]:])

                print(f'epoch|iter: {epoch+1 :4d} | {i :5d} / {len(train_loader) :6d}', end = "  ")
                print(f'({(i*100)/len(train_loader) :0>5.1f}%)', end="  ")
                print(f'D loss: {mean_d_loss :0>9.6f}', end="  ")
                print(f'G loss: {mean_g_loss :0>9.6f}', end="  ")
                print(f'generatorP: {mean_gp :0>.6f}')

                try:
                    # Log metrics to Weights & Biases ............................
                    wandb.log(
                        {
                        "d_loss":   mean_d_loss,
                        "d_r_loss": mean_d_r_loss,
                        "d_f_loss": mean_d_f_loss,
                        "g_loss":   mean_g_loss,
                        "gp":       mean_gp,
                        "epoch":    epoch+1,
                        }
                    )
                except Exception as ex:
                    print(f'An exception of type {type(ex).__name__} occurred. Arguments:\n{ex.args!r}')

                # Save intermediate generated images ....................................
                generator.eval()
                f_imgs_sample = (generator(z_sample).data + 1) / 2.0

                save_dir = f'./results/{config["experiment_name"]}'
                save_image(
                    f_imgs_sample,
                    f'{save_dir}/{config["experiment_name"]}_epoch{str(epoch+1).zfill(3)}_iteration{str(i+1).zfill(5)}.png', 
                    nrow=5
                )

        te        = time.time()
        texec_sec = te - ts
        texec_str = time_format(texec_sec)
        print(f'Epoch training time: {texec_str}')
        results['epoch_training_time'].append(texec_sec)

        try:
            wandb.log(
                {
                "epoch_training_time_sec": texec_sec,
                }
            )
        except Exception as ex:
            print(f'An exception of type {type(ex).__name__} occurred. Arguments:\n{ex.args!r}')

        if ((epoch+1) % config["checkp_interval"] == 0) or ((epoch+1) == config['epochs']):
            file_save_model = f'models/checkpoints/{config["experiment_name"]}_{str(epoch+1).zfill(3)}.pth'
            save_model_and_results(
                discriminator,
                generator,
                d_optimizer,
                g_optimizer,
                results,
                epoch,
                config,
                file_save_model,
            )


In [ ]:
# Create an empty dictionary to store the training results .................
results = {
    'd_loss':              [],
    'd_r_loss':            [],
    'd_f_loss':            [],
    'g_loss':              [],
    'gp':                  [],
    'epoch_training_time': [],
}

# Instantiate the training dataloader ......................................

train_loader, _= get_loader(
    config["crop_size"],
    config["image_size"],
    config["batch_size"],
    train_dir,
)

In [ ]:
# =========================================================================
# Train the model from the beginning
# =========================================================================

if LOAD_TRAINED_MODEL == False and SKIP_TRAIN_MODEL == False:

    train_dragan(
        discriminator,
        generator,
        d_optimizer,
        g_optimizer,
        train_loader,
        0,
        config,
        results,
        device,
    )

# =========================================================================
# Load the saved models
# =========================================================================
# discriminator, generator, d_optimizer, g_optimizer, file_name, device

elif LOAD_TRAINED_MODEL == True:

    file_save_model = f'models/{config["experiment_name"]}.pth'
    results, start_epoch, _ = load_model(
        discriminator,
        generator,
        d_optimizer,
        g_optimizer,
        file_save_model,
        device,
    )

    # ---------------------------------------------------------------------
    # Continue training of the loaded models
    # ---------------------------------------------------------------------

    if SKIP_TRAIN_MODEL == False:

        train_dragan(
            discriminator,
            generator,
            d_optimizer,
            g_optimizer,
            train_loader,
            start_epoch,
            config,
            results,
            device,
        )

## Generate grids of images with the fully trained generator

In [ ]:
def generate_grid_images(generator, num_grids, grid_W_H, config, device):

    grid_size = grid_W_H ** 2
    assert config["batch_size"] >= grid_size, f'Grid size must be less or equal to batch size={config["batch_size"]}'

    generator.eval()

    with torch.inference_mode():

        for num in range(num_grids):

            # Generate a set of latent vectors
            noise = torch.randn(config['batch_size'], config['z_dim'], device=device)

            # Generate a set of fake images with G
            fake = generator(noise).detach().cpu()

            if(config['batch_size'] > grid_size):
                fake = fake[:grid_size]

            # Create a grid with the generated iamges
            grid = make_grid(fake, padding=2, normalize=True)
            grid = grid.permute(1, 2, 0)
            grid = grid.numpy()

            # Display the grid of images
            _ = plt.figure(figsize=(10, 10), constrained_layout=True)
            plt.imshow(grid)

            # Save the grid of images as a PNG file
            file_png = f'results/{config["experiment_name"]}/{config["experiment_name"]}_generated_final_{str(num+1).zfill(3)}.png'
            plt.imsave(file_png, grid)

In [ ]:
generate_grid_images(generator, 8, 8, config, device)

In [ ]:
# Mark the Weights & Bias run as finished
wandb.finish()